# Day 1 — Backend Architecture & REST Principles

---

Before we write a single line of FastAPI code, we need to understand **what a backend actually does** and **how the web talks to it**.

## What is a Backend?

A **backend** is the server-side part of an application. It:

- Listens for requests from clients (browsers, mobile apps, other servers)
- Talks to databases, files, or other services
- Sends back responses (usually JSON)

The **frontend** is what the user sees. The **backend** is the engine behind it.

## Client–Server Model

```
   CLIENT                              SERVER (backend)
   ------                              ----------------
   Browser  ----  HTTP request  --->   FastAPI app
            <---  HTTP response ----   (talks to DB, returns JSON)
```

Every interaction is a **request → response** cycle.
The server doesn't push data on its own (without websockets); the client always asks first.

## Anatomy of an HTTP Request

An HTTP request has four key parts:

| Part | Example |
|------|---------|
| **Method** | `GET`, `POST`, `PUT`, `DELETE` |
| **URL** | `https://api.example.com/users/42` |
| **Headers** | `Content-Type: application/json`, `Authorization: Bearer ...` |
| **Body** | `{"name": "Alice"}` (only for POST/PUT/PATCH) |

A response also has:
- **Status code** (e.g. `200`, `404`)
- **Headers**
- **Body** (usually JSON)

## HTTP Methods (Verbs)

Each method has a specific intent:

| Method | Purpose | Example |
|--------|---------|---------|
| `GET` | **Read** data | `GET /users/1` |
| `POST` | **Create** new resource | `POST /users` with body |
| `PUT` | **Replace** entire resource | `PUT /users/1` with full body |
| `PATCH` | **Update** part of a resource | `PATCH /users/1` with partial body |
| `DELETE` | **Remove** a resource | `DELETE /users/1` |

> **Rule of thumb:** `GET` should never modify data. `POST` always creates.

## Calling an API with `requests`

Python's `requests` library is the easiest way to make HTTP calls.

We'll use [jsonplaceholder.typicode.com](https://jsonplaceholder.typicode.com) — a free fake REST API for testing.

In [ ]:
import requests

response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Body:", response.json())

### Breaking down the response

- `response.status_code` — the numeric result (200 = OK)
- `response.headers` — a dict-like of response headers
- `response.json()` — parses the body as JSON into a Python dict
- `response.text` — the raw body string

## HTTP Status Codes

Status codes are grouped by the first digit:

| Range | Meaning | Common codes |
|-------|---------|---------------|
| **2xx** | Success | `200 OK`, `201 Created`, `204 No Content` |
| **3xx** | Redirect | `301 Moved Permanently`, `304 Not Modified` |
| **4xx** | Client error (your fault) | `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `422 Unprocessable Entity` |
| **5xx** | Server error (their fault) | `500 Internal Server Error`, `502 Bad Gateway`, `503 Service Unavailable` |

> If you remember only a few: **200, 201, 400, 401, 404, 500**.

In [ ]:
# Trigger a 404 by asking for a post that doesn't exist
response = requests.get("https://jsonplaceholder.typicode.com/posts/9999")
print("Status:", response.status_code)
print("Body:", response.text)

## POST — Creating Data

A `POST` request sends a body. With `requests`, pass `json=...` and it serializes for you and sets the right header.

In [ ]:
new_post = {
    "title": "Hello backend",
    "body": "This is my first POST request",
    "userId": 1
}

response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json=new_post
)

print("Status:", response.status_code)   # 201 = Created
print("Created:", response.json())

## PUT — Replacing Data

In [ ]:
updated = {"id": 1, "title": "Updated", "body": "Replaced body", "userId": 1}

response = requests.put(
    "https://jsonplaceholder.typicode.com/posts/1",
    json=updated
)
print("Status:", response.status_code)
print("Body:", response.json())

## DELETE — Removing Data

In [ ]:
response = requests.delete("https://jsonplaceholder.typicode.com/posts/1")
print("Status:", response.status_code)   # 200 OK

## What is REST?

**REST** = **Re**presentational **S**tate **T**ransfer.
It's an architectural style for designing networked APIs. Not a protocol, not a library — a set of conventions.

The core REST constraints (the ones you actually need):

1. **Client–Server** — separation of concerns.
2. **Stateless** — each request contains everything the server needs. No sessions on the server.
3. **Uniform interface** — use standard HTTP methods consistently.
4. **Resource-based** — URLs identify **things** (resources), not actions.

## URL Design Rules

Good RESTful URLs follow simple rules:

| Rule | Bad | Good |
|------|-----|------|
| Use **nouns**, not verbs | `/getUsers` | `/users` |
| Use **plural** for collections | `/user/1` | `/users/1` |
| Use **hierarchy** for relationships | `/userPosts?u=1` | `/users/1/posts` |
| Use **query params** for filters | `/searchPosts?q=...` | `/posts?author=alice` |
| **Lowercase + hyphens** | `/UserPosts` | `/user-posts` |

Example of a clean resource hierarchy:
```
GET    /users               # list users
GET    /users/42            # one user
POST   /users               # create a user
PUT    /users/42            # replace user 42
DELETE /users/42            # delete user 42
GET    /users/42/posts      # all posts by user 42
GET    /users/42/posts/7    # one specific post
```

## Stateless — Why It Matters

Each HTTP request is **independent**. The server doesn't remember what you did last time.

If you need to identify a user, you send a token (e.g. `Authorization: Bearer ...`) **with every request**.

This lets us scale horizontally — any server can handle any request.

In [ ]:
# Headers example: sending an auth token (this endpoint doesn't require one, just illustrating)
headers = {
    "Authorization": "Bearer fake-token-123",
    "User-Agent": "bootcamp-demo/1.0"
}

response = requests.get(
    "https://jsonplaceholder.typicode.com/posts/1",
    headers=headers
)
print("Status:", response.status_code)
print("Body:", response.json())

## Putting It Together — A Mini REST Client

Let's wrap the four basic operations into a single helper function.

In [ ]:
BASE = "https://jsonplaceholder.typicode.com"

def call(method: str, path: str, body: dict | None = None) -> None:
    url = f"{BASE}{path}"
    response = requests.request(method, url, json=body)
    print(f"{method} {path} -> {response.status_code}")
    try:
        print("  body:", response.json())
    except Exception:
        print("  body:", response.text[:80])

call("GET", "/posts/1")
call("POST", "/posts", {"title": "hi", "body": "x", "userId": 1})
call("PUT", "/posts/1", {"id": 1, "title": "new", "body": "y", "userId": 1})
call("DELETE", "/posts/1")

## Quick Recap

- A **backend** listens for HTTP requests and returns responses.
- HTTP methods map to actions: `GET` (read), `POST` (create), `PUT/PATCH` (update), `DELETE` (remove).
- Status codes: **2xx** success, **4xx** client error, **5xx** server error.
- **REST** = resource-based URLs (nouns, plural, hierarchical) + standard HTTP methods + stateless.
- The `requests` library is your tool for *calling* APIs. Next: we'll *build* one with FastAPI.